# Week 61, Day 1: Factor analysis: shared variability against private noise

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Factor analysis by EM

Complete the E step (beta and E[zz]) and the M step (L and Psi) of fa_em, fit two factors to 400 trials of 20 neurons, and compare the largest principal angle between the true loading subspace and the subspaces from factor analysis and from the top two principal components.

In [ ]:
import numpy as np
from scipy.linalg import subspace_angles

def gauss_ll(S, C, n):
    sign, logdet = np.linalg.slogdet(C)
    return -0.5 * n * (len(S) * np.log(2 * np.pi) + logdet + np.trace(np.linalg.solve(C, S)))

def fa_em(X, q, iters=500, seed=0):
    n, d = X.shape
    Xc = X - X.mean(0); S = Xc.T @ Xc / n
    g = np.random.default_rng(seed)
    L = g.normal(size=(d, q)) * np.sqrt(np.diag(S).mean() / q); Psi = np.diag(S).copy()
    hist = []
    for _ in range(iters):
        C = L @ L.T + np.diag(Psi)
        hist.append(gauss_ll(S, C, n))
        beta = ...                                    # q x d
        Ezz = ...
        L = S @ beta.T @ np.linalg.inv(Ezz)
        Psi = np.maximum(np.diag(S - L @ beta @ S), 1e-6)
        if len(hist) > 1 and hist[-1] - hist[-2] < 1e-7:
            break
    return L, Psi, hist

g1 = np.random.default_rng(611)
L_true = g1.normal(size=(20, 2))
psi_true = g1.uniform(0.3, 1.0, 20); psi_true[[3, 11]] = 12.0      # two very noisy neurons
Z = g1.normal(size=(400, 2))
X = Z @ L_true.T + g1.normal(size=(400, 20)) * np.sqrt(psi_true)
L_fa, Psi_fa, fa_ll = fa_em(X, 2)
Xc = X - X.mean(0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
angle_fa = np.degrees(subspace_angles(L_true, L_fa).max())
angle_pca = np.degrees(subspace_angles(L_true, Vt[:2].T).max())
print(f"EM: {len(fa_ll)} iterations, log-likelihood {fa_ll[0]:.0f} -> {fa_ll[-1]:.0f}")
print(f"largest angle to the true subspace: factor analysis {angle_fa:.1f} degrees, PCA {angle_pca:.1f} degrees")

In [ ]:
import numpy as np
def test_ll():
    "the log-likelihood never decreases during EM"
    assert all(b >= a - 1e-6 for a, b in zip(fa_ll, fa_ll[1:]))
def test_subspace():
    "factor analysis recovers the shared subspace better than PCA (smaller largest principal angle)"
    assert angle_fa < 15 and angle_fa < angle_pca - 10

test_ll()
test_subspace()
print("All checks passed")

### Exercise: Cross-validated number of factors

Split the trials into 300 for training and 100 for testing. For q from 1 to 5 fit factor analysis on the training trials and compute the test log-likelihood; take the smallest q within 2 units of the best. Also compute it for probabilistic PCA with q = 2, whose maximum-likelihood fit uses the top two principal components with noise variance equal to the mean of the remaining eigenvalues.

In [ ]:
tr, te = np.arange(300), np.arange(300, 400)
mu_tr = X[tr].mean(0)
Xte = X[te] - mu_tr
S_te = Xte.T @ Xte / len(te)
heldout = {}
for q in range(1, 6):
    Lq, Pq, _ = fa_em(X[tr], q)
    heldout[q] = ...
best_ll = max(heldout.values())
q_best = min(q for q in heldout if heldout[q] >= best_ll - 2)   # smallest q within 2 units of the best

S_tr = np.cov(X[tr].T, bias=True)
ev, U = np.linalg.eigh(S_tr); ev, U = ev[::-1], U[:, ::-1]
sigma2 = ev[2:].mean()
L_ppca = U[:, :2] * np.sqrt(ev[:2] - sigma2)
heldout_ppca = gauss_ll(S_te, L_ppca @ L_ppca.T + sigma2 * np.eye(20), len(te))
print("held-out log-likelihood by number of factors:", {q: round(float(v), 1) for q, v in heldout.items()}, "-> q =", q_best)
print(f"probabilistic PCA with 2 components: {heldout_ppca:.1f}")

In [ ]:
import numpy as np
def test_q():
    "held-out likelihood chooses 2 factors"
    assert q_best == 2
def test_fa_vs_ppca():
    "at two factors, factor analysis has higher held-out likelihood than isotropic-noise PCA"
    assert heldout[2] > heldout_ppca

test_q()
test_fa_vs_ppca()
print("All checks passed")

### Exercise: Percent shared variance

Compute the true and estimated fraction of shared variance for each neuron, using the factor analysis fit to all 400 trials.

In [ ]:
shared_true = (L_true ** 2).sum(1) / ((L_true ** 2).sum(1) + psi_true)
shared_hat = ...
print("true:     ", np.round(shared_true, 2))
print("estimated:", np.round(shared_hat, 2))
print(f"mean percent shared variance: true {100 * shared_true.mean():.1f}%, estimated {100 * shared_hat.mean():.1f}%")

In [ ]:
import numpy as np
def test_shared():
    "estimated shared fractions correlate with the truth above 0.9, and the two noisy neurons have the lowest"
    assert np.corrcoef(shared_hat, shared_true)[0, 1] > 0.9 and set(np.argsort(shared_hat)[:2]) == {3, 11}

test_shared()
print("All checks passed")